In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/raksharajput0488/medical-insurance/medical_insurance.csv


In [2]:
import pandas as pd
df = pd.read_csv('/kaggle/input/datasets/raksharajput0488/medical-insurance/medical_insurance.csv')
df.head()
print(df.isnull().sum())  # check for missing values
print(df.describe())      # sanity check on numeric ranges

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64
               age          bmi     children       charges
count  1338.000000  1338.000000  1338.000000   1338.000000
mean     39.207025    30.663397     1.094918  13270.422265
std      14.049960     6.098187     1.205493  12110.011237
min      18.000000    15.960000     0.000000   1121.873900
25%      27.000000    26.296250     0.000000   4740.287150
50%      39.000000    30.400000     1.000000   9382.033000
75%      51.000000    34.693750     2.000000  16639.912515
max      64.000000    53.130000     5.000000  63770.428010


In [3]:
def bmi_category(bmi):
    if bmi < 18.5:
        return 'Underweight'
    elif bmi < 25:
        return 'Normal'
    elif bmi < 30:
        return 'Overweight'
    else:
        return 'Obese'

df['bmi_category'] = df['bmi'].apply(bmi_category)

In [4]:
def age_group(age):
    if age < 25:
        return '18-24'
    elif age < 35:
        return '25-34'
    elif age < 45:
        return '35-44'
    elif age < 55:
        return '45-54'
    else:
        return '55+'

df['age_group'] = df['age'].apply(age_group)

In [5]:
import sqlite3
conn = sqlite3.connect(':memory:')
df.to_sql('insurance', conn, index=False)

# Q1: Smoker vs non-smoker average charges
q1 = """
SELECT smoker, COUNT(*) as total, ROUND(AVG(charges),2) as avg_charges
FROM insurance GROUP BY smoker
"""
print(pd.read_sql(q1, conn))

# Q2: BMI category vs charges
q2 = """
SELECT bmi_category, COUNT(*) as total, ROUND(AVG(charges),2) as avg_charges
FROM insurance GROUP BY bmi_category ORDER BY avg_charges DESC
"""
print(pd.read_sql(q2, conn))

# Q3: Age group vs charges
q3 = """
SELECT age_group, COUNT(*) as total, ROUND(AVG(charges),2) as avg_charges
FROM insurance GROUP BY age_group ORDER BY age_group
"""
print(pd.read_sql(q3, conn))

# Q4: Region vs charges
q4 = """
SELECT region, COUNT(*) as total, ROUND(AVG(charges),2) as avg_charges
FROM insurance GROUP BY region ORDER BY avg_charges DESC
"""
print(pd.read_sql(q4, conn))

  smoker  total  avg_charges
0     no   1064      8434.27
1    yes    274     32050.23
  bmi_category  total  avg_charges
0        Obese    707     15552.34
1   Overweight    386     10987.51
2       Normal    225     10409.34
3  Underweight     20      8852.20
  age_group  total  avg_charges
0     18-24    278      9011.34
1     25-34    271     10352.39
2     35-44    260     13134.17
3     45-54    287     15853.93
4       55+    242     18513.28
      region  total  avg_charges
0  southeast    364     14735.41
1  northeast    324     13406.38
2  northwest    325     12417.58
3  southwest    325     12346.94


In [6]:
q5 = """
SELECT smoker, bmi_category, COUNT(*) as total, ROUND(AVG(charges),2) as avg_charges
FROM insurance GROUP BY smoker, bmi_category ORDER BY avg_charges DESC
"""
print(pd.read_sql(q5, conn))

  smoker bmi_category  total  avg_charges
0    yes        Obese    145     41557.99
1    yes   Overweight     74     22495.87
2    yes       Normal     50     19942.22
3    yes  Underweight      5     18809.82
4     no        Obese    562      8842.69
5     no   Overweight    312      8257.96
6     no       Normal    175      7685.66
7     no  Underweight     15      5532.99


In [7]:
# Estimate potential savings if smokers had non-smoker average charges
smoker_avg = df[df['smoker']=='yes']['charges'].mean()
nonsmoker_avg = df[df['smoker']=='no']['charges'].mean()
num_smokers = df[df['smoker']=='yes'].shape[0]

potential_savings_per_person = smoker_avg - nonsmoker_avg
total_potential_savings = potential_savings_per_person * num_smokers

print(f"Average charge gap (smoker vs non-smoker): ${potential_savings_per_person:,.2f}")
print(f"Number of smokers in dataset: {num_smokers}")
print(f"Estimated total cost gap across all smokers: ${total_potential_savings:,.2f}")

Average charge gap (smoker vs non-smoker): $23,615.96
Number of smokers in dataset: 274
Estimated total cost gap across all smokers: $6,470,774.01


In [8]:
# Full segment breakdown to find the extremes
q6 = """
SELECT smoker, bmi_category, age_group, COUNT(*) as total, ROUND(AVG(charges),2) as avg_charges
FROM insurance GROUP BY smoker, bmi_category, age_group
HAVING total >= 10
ORDER BY avg_charges DESC
"""
print(pd.read_sql(q6, conn))

   smoker bmi_category age_group  total  avg_charges
0     yes        Obese       55+     26     46979.92
1     yes        Obese     45-54     29     45434.73
2     yes        Obese     35-44     31     40560.01
3     yes        Obese     25-34     26     39865.23
4     yes        Obese     18-24     33     36150.52
5     yes   Overweight       55+     12     28642.79
6     yes   Overweight     45-54     14     24073.74
7     yes   Overweight     35-44     16     24005.60
8     yes       Normal     45-54     12     23952.73
9     yes       Normal     35-44     13     19450.44
10    yes   Overweight     25-34     14     19333.95
11    yes   Overweight     18-24     18     18288.00
12    yes       Normal     25-34     13     17073.48
13     no        Obese       55+    121     14310.25
14     no       Normal       55+     22     13805.95
15     no   Overweight       55+     55     13690.08
16     no   Overweight     45-54     62     11787.79
17     no        Obese     45-54    133     11